# 17 — GA-based Counterfactual Search — BPIC17

Uses a genetic algorithm with constraint-preserving operators (Guidotti et al. 2024 adapted)
to generate counterfactual explanations for next-activity prediction.

Loads pre-mined constraints from 16a. No VAE needed (direct sequence mutation).

In [1]:
import sys
import os
from pathlib import Path

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
import torch

# --- Load dataset + prediction model ---
data_path = _current / 'encoded_data' / 'BPIC_2017_all_5_test.pkl'
full_dataset = torch.load(data_path, weights_only=False)
dataset = full_dataset

sample = dataset[0]
n_cat = len(sample[0])
n_num = len(sample[1])
seq_len = sample[0][0].shape[0]
print(f'Dataset: {len(dataset)} sequences, {n_cat} cat, {n_num} num, seq_len={seq_len}')

from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

from src.interpretability.config.bpic17_config import CONFIG
CONFIG.use_improved = False  # set True for the improved variant
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(CONFIG.get_model_path()), dropout=0.0)
model.eval()
print(f'Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters')

# --- TensorDecoder + activity vocabulary ---
from src.interpretability.utils.tensor_decoder import TensorDecoder

decoder = TensorDecoder(full_dataset)

ACTIVITY_FEATURE = 'concept:name'
activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
max_idx = max(activity_idx_to_label.keys())
activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]
eos_idx = next(i for i, name in enumerate(activity_names) if name == 'EOS')

print(f'Activity vocabulary ({len(activity_names)}), EOS={eos_idx}')
print(f'Cat features: {decoder.cat_features}')

In [3]:
# Load pre-mined constraints and parse data conditions
import pickle
from src.interpretability.perturbation_methods.revised_plus.rum_mpdeclare import parse_data_condition_string

constraints_pkl_path = _current / 'encoded_data' / 'bpic17_constraints.pkl'
with open(constraints_pkl_path, 'rb') as f:
    constraints_data = pickle.load(f)

all_constraints = constraints_data['all']
data_conditions_raw = constraints_data.get('data_conditions', {})

# Parse data condition strings into structured DataCondition objects
data_conditions = {}
n_skipped = 0
for dc, raw_str in data_conditions_raw.items():
    if isinstance(raw_str, str):
        parsed = parse_data_condition_string(
            raw_str,
            cat_feature_names=decoder.cat_features,
            cat_label_to_idx=decoder.label_to_idx,
        )
        if parsed is not None:
            data_conditions[dc] = parsed
        else:
            n_skipped += 1
    else:
        # Already a DataCondition object
        data_conditions[dc] = raw_str

print(f'Constraints: {len(all_constraints)} (from {constraints_pkl_path.name})')
print(f'Data conditions: {len(data_conditions)} parsed, {n_skipped} skipped (unresolvable features)')

/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/lark/utils.py:163: DeprecationWarning: module 'sre_parse' is deprecated
  import sre_parse
/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/lark/utils.py:164: DeprecationWarning: module 'sre_constants' is deprecated
  import sre_constants


Constraints: 147 (from bpic17_constraints.pkl)
Data conditions: 0 parsed, 0 skipped (unresolvable features)


In [4]:
from src.interpretability.perturbation_methods import (
    GACounterfactual, GACounterfactualConfig, create_ga_counterfactual_for_model,
)

# BPIC17 categorical features:
#   0 = concept:name (28)       — activity
#   1 = Action (7)              — event-level
#   2 = org:resource (151)      — event-level
#   3 = EventOrigin (5)         — event-level
#   4 = lifecycle:transition (9) — event-level
#   5 = case:LoanGoal (16)      — CASE-LEVEL
#   6 = case:ApplicationType (4) — CASE-LEVEL
#   7 = Accepted (5)            — CASE-LEVEL
#   8 = Selected (5)            — CASE-LEVEL
MUTABLE_CAT_INDICES = [0, 2, 5, 6]
CASE_LEVEL_CAT_INDICES = [5, 6]

config = GACounterfactualConfig(
    population_size=200,
    n_generations=50,
    crossover_rate=0.8,
    mutation_rate=0.3,
    tournament_size=5,
    elite_size=5,
    mutable_cat_indices=MUTABLE_CAT_INDICES,
    case_level_cat_indices=CASE_LEVEL_CAT_INDICES,
    w_validity=5.0,
    w_proximity=1.0,
    w_sparsity=1.0,
    w_plausibility=1.0,
    w_conformance=2.0,
    top_k=5,
    diversity_threshold=0.3,
    activity_feature=ACTIVITY_FEATURE,
    verbose=True,
)

ga = create_ga_counterfactual_for_model(
    model=model,
    dataset=dataset,
    activity_names=activity_names,
    config=config,
    constraints=all_constraints,
    cat_feature_names=decoder.cat_features,
    data_conditions=data_conditions,
)
print(f'\nGA Counterfactual ready')

GA Counterfactual: 6301 training sequences, 147 constraints
  Mutable features: concept:name(vocab=28), org:resource(vocab=151), case:LoanGoal(vocab=16), case:ApplicationType(vocab=4)

GA Counterfactual ready


In [5]:
import numpy as np
import pandas as pd

# === Scan dataset for candidate sequences ===
N_CANDIDATES = 50
scan_limit = min(500, len(dataset))

candidates = []
seen_cases = set()
for i in range(scan_limit):
    cat_t, num_t, case_id = dataset[i]
    act = cat_t[0]

    if case_id in seen_cases:
        continue
    seen_cases.add(case_id)

    trace_len = int((act != 0).sum().item())
    act_seq = [activity_names[a.item()] for a in act if a.item() != 0]

    candidates.append({
        'dataset_idx': i,
        'case_id': case_id,
        'trace_len': trace_len,
        'activities': ' -> '.join(act_seq),
    })

    if len(candidates) >= N_CANDIDATES:
        break

df_candidates = pd.DataFrame(candidates)
print(f'Found {len(candidates)} unique cases (scanned {scan_limit})')
print('Set SELECTED_ROW and PREFIX_LEN below to pick a case and prefix length.\n')
display(df_candidates)

Found 14 unique cases (scanned 500)
Set SELECTED_ROW and PREFIX_LEN below to pick a case and prefix length.



,dataset_idx,case_id,trace_len,activities
0,0,Application_1000086665,5,A_Create Application -> A_Submitted -> W_Handl...
1,24,Application_1000806256,5,A_Create Application -> W_Complete application...
2,55,Application_1001114274,5,A_Create Application -> A_Submitted -> W_Handl...
3,126,Application_1001866944,5,A_Create Application -> A_Submitted -> W_Handl...
4,179,Application_1002013470,5,A_Create Application -> W_Complete application...
5,219,Application_1002032547,5,A_Create Application -> A_Submitted -> W_Handl...
6,243,Application_1002348901,5,A_Create Application -> A_Submitted -> W_Handl...
7,266,Application_1002442935,5,A_Create Application -> A_Submitted -> W_Handl...
8,290,Application_1002485344,5,A_Create Application -> A_Submitted -> W_Handl...
9,372,Application_100305707,5,A_Create Application -> A_Submitted -> W_Handl...


In [6]:
# ============================================
# SELECT A CASE AND PREFIX LENGTH
# ============================================
SELECTED_ROW = 0   # row index in the candidates table above
PREFIX_LEN = 5     # how many events to use as prefix (1 .. trace_len)

# --- Load and truncate to prefix ---
selected = df_candidates.iloc[SELECTED_ROW]
test_idx = selected['dataset_idx']
trace_len = selected['trace_len']
cat_tuple_full, num_tuple_full, case_id = dataset[test_idx]

prefix_len = max(1, min(PREFIX_LEN, trace_len))
if prefix_len != PREFIX_LEN:
    print(f'Note: PREFIX_LEN clamped to {prefix_len} (trace has {trace_len} events)')

# Build left-padded prefix tensors
pad_len = seq_len - prefix_len
cat_tuple = []
for c in cat_tuple_full:
    t = torch.zeros_like(c)
    src_start = seq_len - trace_len
    t[pad_len:] = c[src_start:src_start + prefix_len]
    cat_tuple.append(t)
cat_tuple = tuple(cat_tuple)

num_tuple = []
for n in num_tuple_full:
    t = torch.zeros_like(n)
    src_start = seq_len - trace_len
    t[pad_len:] = n[src_start:src_start + prefix_len]
    num_tuple.append(t)
num_tuple = tuple(num_tuple)

# Show prediction for this prefix
prefix_acts = [activity_names[cat_tuple[0][j].item()] for j in range(pad_len, seq_len)]
full_acts = [activity_names[a.item()] for a in cat_tuple_full[0] if a.item() != 0]

cat_in = [c.unsqueeze(0) for c in cat_tuple]
num_in = [n.unsqueeze(0) for n in num_tuple]
with torch.no_grad():
    preds = model((cat_in, num_in))[0]
    logits = preds[0][f'{ACTIVITY_FEATURE}_mean'][0]
    p = torch.softmax(logits, dim=-1)
    top_p, top_idx = p.max(dim=-1)

print(f'Case: {case_id}')
print(f'Full trace ({trace_len}): {" -> ".join(full_acts)}')
print(f'Prefix ({prefix_len}/{trace_len}):  {" -> ".join(prefix_acts)}')
print(f'Predicted next: {activity_names[top_idx.item()]} (p={top_p.item():.3f})')
print()
df_orig = decoder.decode_sequence(cat_tuple, num_tuple, case_id=case_id)
display(df_orig)

Case: Application_1000086665
Full trace (5): A_Create Application -> A_Submitted -> W_Handle leads -> W_Handle leads -> W_Complete application
Prefix (5/5):  A_Create Application -> A_Submitted -> W_Handle leads -> W_Handle leads -> W_Complete application
Predicted next: A_Concept (p=0.754)



,concept:name,Action,org:resource,EventOrigin,lifecycle:transition,case:LoanGoal,case:ApplicationType,Accepted,Selected,case_elapsed_time,event_elapsed_time,day_in_week,seconds_in_day,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,MonthlyCost,CreditScore,Case ID
0,A_Create Application,Created,User_1,Application,complete,"Other, see explanation",New credit,<none>,<none>,0.0000,50897.105469,2.0,57441.0,5000.0,8383.772461,82.958527,280.393005,319.191345,Application_1000086665
1,A_Submitted,statechange,User_1,Application,complete,"Other, see explanation",New credit,<none>,<none>,0.0625,0.062500,2.0,57441.0,5000.0,8383.772461,82.958527,280.393005,319.191345,Application_1000086665
2,W_Handle leads,Created,User_1,Workflow,schedule,"Other, see explanation",New credit,<none>,<none>,0.3125,0.230469,2.0,57441.0,5000.0,8383.772461,82.958527,280.393005,319.191345,Application_1000086665
3,W_Handle leads,Deleted,User_1,Workflow,withdraw,"Other, see explanation",New credit,<none>,<none>,66.6250,66.324219,2.0,57508.0,5000.0,8383.772461,82.958527,280.393005,319.191345,Application_1000086665
4,W_Complete application,Created,User_1,Workflow,schedule,"Other, see explanation",New credit,<none>,<none>,66.6250,0.007812,2.0,57508.0,5000.0,8383.772461,82.958527,280.393005,319.191345,Application_1000086665


In [7]:
# === Run GA counterfactual search ===
cat_tensors = [c.unsqueeze(0) for c in cat_tuple]
num_tensors = [n.unsqueeze(0) for n in num_tuple]

explanation = ga.explain(cat_tensors, num_tensors, target_class=None)
print(explanation)

Original: A_Concept (p=0.754), prefix_len=5, mutable=['cat[0]', 'cat[2]', 'cat[5]', 'cat[6]']
  Gen   0: best_fit=0.1000, valid=29/200, total_valid=29
  Gen  10: best_fit=0.1000, valid=194/200, total_valid=1848
  Gen  20: best_fit=0.1000, valid=191/200, total_valid=3763
  Gen  30: best_fit=0.1000, valid=197/200, total_valid=5721
  Gen  40: best_fit=0.1000, valid=188/200, total_valid=7637
  Gen  49: best_fit=0.1000, valid=194/200, total_valid=9363
GA Counterfactual Explanation
Original: A_Concept (idx=4, p=0.754)
Prefix length: 5 events
Mutable features: cat indices [0, 2, 5, 6]
Constraints: 121
Search: 50 generations, 10000 evaluated, 9363 valid, 22.3s

Top 5 counterfactuals:
  [1] -> W_Complete application (p=0.540) | prox=0.050 sparse=1 plaus=0.000 conf=1.00 fit=0.1000 gen=0
  [2] -> W_Complete application (p=0.621) | prox=0.150 sparse=3 plaus=0.200 conf=1.00 fit=0.5000 gen=36
  [3] -> W_Complete application (p=0.561) | prox=0.150 sparse=3 plaus=0.200 conf=1.00 fit=0.5000 gen=42
  [4

In [8]:
# === Display counterfactuals ===
if not explanation.counterfactuals:
    print('No counterfactuals found. Try increasing n_generations or population_size.')
else:
    # --- Original ---
    orig_acts_str = ' -> '.join(activity_names[a] for a in explanation.original_activity_sequence)
    print('=' * 100)
    print(f'ORIGINAL  \u2014  Case: {case_id}')
    print(f'  {orig_acts_str}  ->  [{explanation.original_prediction_name}] (p={explanation.original_probability:.3f})')
    print('=' * 100)

    # Build original decoded values per mutable feature for diff
    orig_decoded = {}
    for ci in ga.mutable_cat_indices:
        feat_name = decoder.cat_features[ci]
        orig_tensor = cat_tuple[ci]
        orig_vals = [
            decoder.decode_categorical_value(feat_name, orig_tensor[j].item())
            for j in range(pad_len, seq_len)
        ]
        orig_decoded[ci] = orig_vals

    # --- Each counterfactual ---
    for i, cf in enumerate(explanation.counterfactuals):
        print(f'\n{"\u2014" * 100}')
        # Activity sequence + new prediction
        cf_acts_str = ' -> '.join(activity_names[a] for a in cf.activity_sequence)
        print(f'CF #{i+1}:  {cf_acts_str}  ->  [{cf.counterfactual_prediction_name}] (p={cf.counterfactual_probability:.3f})')
        print(f'  fitness={cf.fitness:.4f}  prox={cf.proximity:.3f}  sparse={cf.sparsity}  conf={cf.conformance:.2f}  gen={cf.generation}')

        # Show only changed features
        cf_cat_squeezed = [t.squeeze(0) for t in cf.cat_sequence]
        changes = []
        for ci in ga.mutable_cat_indices:
            feat_name = decoder.cat_features[ci]
            cf_tensor = cf_cat_squeezed[ci]
            for pos_idx, pos in enumerate(range(pad_len, seq_len)):
                orig_val = orig_decoded[ci][pos_idx]
                cf_val = decoder.decode_categorical_value(feat_name, cf_tensor[pos].item())
                if orig_val != cf_val:
                    changes.append({
                        'position': pos_idx + 1,
                        'feature': feat_name,
                        'original': orig_val,
                        'counterfactual': cf_val,
                    })

        if changes:
            print(f'  Changes ({len(changes)}):')
            display(pd.DataFrame(changes))
        else:
            print('  No feature changes (prediction changed via numerical context)')

ORIGINAL  —  Case: Application_1000086665
  A_Create Application -> A_Submitted -> W_Handle leads -> W_Handle leads -> W_Complete application  ->  [A_Concept] (p=0.754)

————————————————————————————————————————————————————————————————————————————————————————————————————
CF #1:  A_Create Application -> A_Submitted -> W_Handle leads -> W_Handle leads -> W_Complete application  ->  [W_Complete application] (p=0.540)
  fitness=0.1000  prox=0.050  sparse=1  conf=1.00  gen=0
  Changes (1):


,position,feature,original,counterfactual
0,5,org:resource,User_1,User_94



————————————————————————————————————————————————————————————————————————————————————————————————————
CF #2:  A_Create Application -> A_Submitted -> W_Handle leads -> A_Pending -> W_Handle leads  ->  [W_Complete application] (p=0.621)
  fitness=0.5000  prox=0.150  sparse=3  conf=1.00  gen=36
  Changes (3):


,position,feature,original,counterfactual
0,4,concept:name,W_Handle leads,A_Pending
1,5,concept:name,W_Complete application,W_Handle leads
2,5,org:resource,User_1,User_76



————————————————————————————————————————————————————————————————————————————————————————————————————
CF #3:  A_Create Application -> A_Submitted -> A_Complete -> W_Handle leads -> W_Handle leads  ->  [W_Complete application] (p=0.561)
  fitness=0.5000  prox=0.150  sparse=3  conf=1.00  gen=42
  Changes (3):


,position,feature,original,counterfactual
0,3,concept:name,W_Handle leads,A_Complete
1,5,concept:name,W_Complete application,W_Handle leads
2,5,org:resource,User_1,User_78



————————————————————————————————————————————————————————————————————————————————————————————————————
CF #4:  A_Create Application -> O_Accepted -> W_Handle leads -> W_Handle leads -> A_Pending  ->  [W_Complete application] (p=0.391)
  fitness=0.7000  prox=0.150  sparse=3  conf=1.00  gen=2
  Changes (3):


,position,feature,original,counterfactual
0,2,concept:name,A_Submitted,O_Accepted
1,5,concept:name,W_Complete application,A_Pending
2,5,org:resource,User_1,User_133



————————————————————————————————————————————————————————————————————————————————————————————————————
CF #5:  A_Create Application -> W_Shortened completion  -> W_Handle leads -> A_Complete -> W_Complete application  ->  [W_Complete application] (p=0.481)
  fitness=0.7000  prox=0.150  sparse=3  conf=1.00  gen=3
  Changes (3):


,position,feature,original,counterfactual
0,2,concept:name,A_Submitted,W_Shortened completion
1,4,concept:name,W_Handle leads,A_Complete
2,5,org:resource,User_1,User_2
